# Step 6 — Evaluate the Model
**Medical Chatbot | Multi-Experiment Evaluation**

| What | Details |
|------|--------|
| Input | Experiment folder + test data |
| Metrics | Test Loss, Perplexity, BLEU, ROUGE-1/2/L |
| Output | `experiments/EXP_NAME/eval_results.json` |

**How to use:** Change `EXP_NAME` in Cell 0, then run Cells 0 → 5 in order. Cell 6 shows a comparison table across all evaluated experiments.

## Cell 0 — Experiment Selection
**Only change this cell** to switch between experiments. All paths are derived automatically.

In [39]:
EXP_NAME = 'exp4_d512_heads2_layers8'   # <- change to evaluate a different experiment

# Derived paths — do not edit
EXP_DIR         = f'../experiments/{EXP_NAME}'
CHECKPOINT_PATH = f'{EXP_DIR}/checkpoints/best_model.pt'
CONFIG_PATH     = f'{EXP_DIR}/config.json'
EVAL_OUT_PATH   = f'{EXP_DIR}/eval_results.json'

print(f'Experiment  : {EXP_NAME}')
print(f'Config      : {CONFIG_PATH}')
print(f'Checkpoint  : {CHECKPOINT_PATH}')
print(f'Results out : {EVAL_OUT_PATH}')

Experiment  : exp4_d512_heads2_layers8
Config      : ../experiments/exp4_d512_heads2_layers8/config.json
Checkpoint  : ../experiments/exp4_d512_heads2_layers8/checkpoints/best_model.pt
Results out : ../experiments/exp4_d512_heads2_layers8/eval_results.json


## Cell 1 — Imports & Setup
Loads config from the selected experiment folder and sets up the device.

In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import json, math, os
import numpy as np
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

try:
    from rouge_score import rouge_scorer
except ImportError:
    os.system('pip install rouge-score')
    from rouge_score import rouge_scorer

# Validate experiment folder exists
assert os.path.isdir(EXP_DIR),          f'Experiment folder not found: {EXP_DIR}'
assert os.path.isfile(CONFIG_PATH),     f'config.json not found: {CONFIG_PATH}'
assert os.path.isfile(CHECKPOINT_PATH), f'Checkpoint not found: {CHECKPOINT_PATH}'

# Load config from experiment folder
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

VOCAB_SIZE  = config['vocab_size']
MAX_SEQ_LEN = config['max_seq_len']
D_MODEL     = config['d_model']
N_HEADS     = config['n_heads']
N_LAYERS    = config['n_layers']
FFN_DIM     = config['ffn_dim']
PAD_ID      = config['pad_id']
EOS_ID      = config['eos_id']
PATIENT_ID  = config['patient_id']
DOCTOR_ID   = config['doctor_id']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')
print(f'Config   : vocab={VOCAB_SIZE}, d_model={D_MODEL}, n_heads={N_HEADS}, n_layers={N_LAYERS}, ffn_dim={FFN_DIM}')

Device   : cuda
Config   : vocab=8000, d_model=512, n_heads=2, n_layers=8, ffn_dim=2048


## Cell 2 — Model Definition & Load Checkpoint
Defines the full MedicalGPT architecture and loads weights from the experiment checkpoint.

The model classes are the same as Step 4 — reproduced here so this notebook is self-contained.

In [42]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len, pad_id, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)
        self.dropout   = nn.Dropout(dropout)
        self.d_model   = d_model
    def forward(self, x):
        seq_len   = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        tok = self.token_emb(x)
        pos = self.pos_emb(positions)
        return self.dropout(tok * math.sqrt(self.d_model) + pos)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.W_q     = nn.Linear(d_model, d_model)
        self.W_k     = nn.Linear(d_model, d_model)
        self.W_v     = nn.Linear(d_model, d_model)
        self.W_o     = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, causal_mask=None, pad_mask=None):
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
        attn = torch.nan_to_num(F.softmax(scores, dim=-1), nan=0.0)
        attn = self.dropout(attn)
        out  = torch.matmul(attn, V)
        return self.W_o(out.transpose(1, 2).contiguous().view(B, T, self.d_model))

class FeedForward(nn.Module):
    def __init__(self, d_model, ffn_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn     = FeedForward(d_model, ffn_dim, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, causal_mask=None, pad_mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), causal_mask, pad_mask))
        x = x + self.ffn(self.norm2(x))
        return x

class MedicalGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, ffn_dim,
                 max_seq_len, pad_id, dropout=0.1):
        super().__init__()
        self.pad_id     = pad_id
        self.embeddings = Embeddings(vocab_size, d_model, max_seq_len, pad_id, dropout)
        self.layers     = nn.ModuleList([
            DecoderLayer(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.norm    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embeddings.token_emb.weight  # weight tying
    def make_causal_mask(self, seq_len, device):
        return torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)
    def forward(self, input_ids, attention_mask=None):
        B, T  = input_ids.shape
        x     = self.embeddings(input_ids)
        cmask = self.make_causal_mask(T, input_ids.device)
        for layer in self.layers:
            x = layer(x, causal_mask=cmask, pad_mask=attention_mask)
        return self.lm_head(self.norm(x))

# Load model from experiment checkpoint
# weights_only=False is required here because the checkpoint contains a config dict (not just tensors)
model = MedicalGPT(VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS, FFN_DIM, MAX_SEQ_LEN, PAD_ID)
ckpt  = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
model.eval()
print(f'Model loaded from: {CHECKPOINT_PATH}')

total_params = sum(p.numel() for p in model.parameters())
print(f'   Parameters: {total_params:,} ({total_params/1e6:.2f}M)')

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.Load('../tokenizer/medical_bpe.model')
print('Tokenizer loaded!')

Model loaded from: ../experiments/exp4_d512_heads2_layers8/checkpoints/best_model.pt
   Parameters: 29,578,240 (29.58M)
Tokenizer loaded!


## Cell 3 — Perplexity on Full Test Set
Runs the model over all 10,130 test samples and computes average cross-entropy loss and perplexity.

**Perplexity** = exp(avg loss). Lower is better — it measures how "surprised" the model is by the test data.

In [43]:
class MedicalDataset(Dataset):
    def __init__(self, tokens_path, masks_path):
        self.tokens = torch.load(tokens_path, weights_only=True)
        self.masks  = torch.load(masks_path,  weights_only=True)
    def __len__(self):
        return len(self.tokens)
    def __getitem__(self, idx):
        return self.tokens[idx], self.masks[idx]

# Load test data
test_dataset = MedicalDataset('../data/test_tokens.pt', '../data/test_masks.pt')
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
print(f'Test samples: {len(test_dataset)}')

# Calculate Perplexity
criterion     = nn.CrossEntropyLoss(ignore_index=PAD_ID)
total_loss    = 0
total_batches = 0

print('Calculating perplexity on full test set...')
with torch.no_grad():
    for i, (tokens, masks) in enumerate(test_loader):
        tokens      = tokens.to(device)
        masks       = masks.to(device)
        input_ids   = tokens[:, :-1]
        target_ids  = tokens[:, 1:]
        input_masks = masks[:, :-1]
        logits = model(input_ids, attention_mask=input_masks)
        loss   = criterion(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))
        total_loss    += loss.item()
        total_batches += 1
        if (i + 1) % 20 == 0:
            print(f'  Batch {i+1}/{len(test_loader)} done...')

avg_loss   = total_loss / total_batches
perplexity = math.exp(avg_loss)
print()
print('=' * 50)
print(f'  Experiment      : {EXP_NAME}')
print(f'  Test Loss       : {avg_loss:.4f}')
print(f'  Test Perplexity : {perplexity:.2f}')
print('=' * 50)

Test samples: 10130
Calculating perplexity on full test set...
  Batch 20/159 done...
  Batch 40/159 done...
  Batch 60/159 done...
  Batch 80/159 done...
  Batch 100/159 done...
  Batch 120/159 done...
  Batch 140/159 done...

  Experiment      : exp4_d512_heads2_layers8
  Test Loss       : 2.9507
  Test Perplexity : 19.12


## Cell 4 — BLEU & ROUGE on Full Test Set

For each test sample:
1. Extract the patient prompt (up to and including `<doctor>` token)
2. Generate a response using the model (temperature=0.7, top-k=50)
3. Compare generated response vs reference doctor response

**BLEU** — measures n-gram overlap between generated and reference text  
**ROUGE** — measures recall of reference text in generated output

In [53]:
print(f'Generating responses for full test set (~{len(test_dataset):,} samples)...')
print('This will take several minutes...\n')

scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoother = SmoothingFunction().method1

references    = []
hypotheses    = []
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

def generate_for_eval(input_ids_list, max_new_tokens=100, temperature=0.7, top_k=50):
    """Autoregressively generate tokens given a prompt (list of token IDs)."""
    input_tensor = torch.tensor([input_ids_list], dtype=torch.long).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if input_tensor.size(1) >= MAX_SEQ_LEN:
                break
            logits      = model(input_tensor)
            next_logits = logits[0, -1, :] / temperature
            top_k_vals, top_k_idx = torch.topk(next_logits, top_k)
            probs       = torch.softmax(top_k_vals, dim=-1)
            next_token  = top_k_idx[torch.multinomial(probs, 1)].view(1, 1)
            if next_token.item() == EOS_ID:
                break
            input_tensor = torch.cat([input_tensor, next_token], dim=1)
    return input_tensor[0].tolist()

# Loop through all test samples
for i in range(len(test_dataset)):
    tokens, masks = test_dataset[i]
    token_list    = tokens.tolist()

    if DOCTOR_ID not in token_list:
        continue
    doctor_pos     = token_list.index(DOCTOR_ID)
    patient_prompt = token_list[:doctor_pos + 1]

    # Extract reference doctor response
    reference_ids = []
    for t in token_list[doctor_pos + 1:]:
        if t in [PAD_ID, EOS_ID]:
            break
        reference_ids.append(t)
    if len(reference_ids) < 3:
        continue

    # Generate model response
    generated_ids  = generate_for_eval(patient_prompt)
    generated_ids  = generated_ids[doctor_pos + 1:]

    reference_text = sp.Decode(reference_ids)
    generated_text = sp.Decode(generated_ids)

    # BLEU
    ref_tokens = reference_text.lower().split()
    hyp_tokens = generated_text.lower().split()
    references.append([ref_tokens])
    hypotheses.append(hyp_tokens)

    # ROUGE
    scores = scorer.score(reference_text, generated_text)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

    if (i + 1) % 500 == 0:
        print(f'  Processed {i+1}/{len(test_dataset)} samples...')

# Final Scores
bleu = corpus_bleu(references, hypotheses, smoothing_function=smoother)
r1   = float(np.mean(rouge1_scores))
r2   = float(np.mean(rouge2_scores))
rl   = float(np.mean(rougeL_scores))

print()
print('=' * 50)
print('     EVALUATION RESULTS — Full Test Set')
print('=' * 50)
print(f'  Experiment        : {EXP_NAME}')
print(f'  Samples evaluated : {len(hypotheses)}')
print(f'  BLEU Score        : {bleu:.4f}  ({bleu*100:.2f}%)')
print(f'  ROUGE-1           : {r1:.4f}  ({r1*100:.2f}%)')
print(f'  ROUGE-2           : {r2:.4f}  ({r2*100:.2f}%)')
print(f'  ROUGE-L           : {rl:.4f}  ({rl*100:.2f}%)')
print('=' * 50)

Generating responses for full test set (~10,130 samples)...
This will take several minutes...

  Processed 500/10130 samples...
  Processed 1000/10130 samples...
  Processed 1500/10130 samples...
  Processed 2000/10130 samples...
  Processed 2500/10130 samples...
  Processed 3000/10130 samples...
  Processed 3500/10130 samples...
  Processed 4000/10130 samples...
  Processed 4500/10130 samples...
  Processed 5000/10130 samples...
  Processed 5500/10130 samples...
  Processed 6000/10130 samples...
  Processed 6500/10130 samples...
  Processed 7000/10130 samples...
  Processed 7500/10130 samples...
  Processed 8000/10130 samples...
  Processed 8500/10130 samples...
  Processed 9000/10130 samples...
  Processed 9500/10130 samples...
  Processed 10000/10130 samples...

     EVALUATION RESULTS — Full Test Set
  Experiment        : exp4_d512_heads2_layers8
  Samples evaluated : 10130
  BLEU Score        : 0.0317  (3.17%)
  ROUGE-1           : 0.2355  (23.55%)
  ROUGE-2           : 0.0444  (4

## Cell 5 — Save Results to Experiment Folder
Saves all metrics as `eval_results.json` inside the experiment folder.

In [54]:
eval_results = {
    'experiment'      : EXP_NAME,
    'config'          : config,
    'test_samples'    : len(hypotheses),
    'test_loss'       : round(avg_loss,   4),
    'test_perplexity' : round(perplexity, 4),
    'bleu'            : round(bleu * 100, 4),
    'rouge1'          : round(r1   * 100, 4),
    'rouge2'          : round(r2   * 100, 4),
    'rougeL'          : round(rl   * 100, 4),
}

with open(EVAL_OUT_PATH, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f'Results saved to: {EVAL_OUT_PATH}')
print(json.dumps(eval_results, indent=2))

Results saved to: ../experiments/exp4_d512_heads2_layers8/eval_results.json
{
  "experiment": "exp4_d512_heads2_layers8",
  "config": {
    "model_path": "../tokenizer/medical_bpe.model",
    "vocab_size": 8000,
    "max_seq_len": 512,
    "batch_size": 32,
    "d_model": 512,
    "n_heads": 2,
    "n_layers": 8,
    "ffn_dim": 2048,
    "pad_id": 0,
    "unk_id": 1,
    "eos_id": 4,
    "patient_id": 5,
    "doctor_id": 6
  },
  "test_samples": 10130,
  "test_loss": 2.9507,
  "test_perplexity": 19.1192,
  "bleu": 3.1682,
  "rouge1": 23.5538,
  "rouge2": 4.4419,
  "rougeL": 14.7713
}


## Cell 6 — Cross-Experiment Comparison Table
Automatically scans all experiment folders for `eval_results.json` and prints a side-by-side comparison.

Run this any time — it picks up whichever experiments have been evaluated so far.

In [55]:
experiments_root = '../experiments'
rows = []

for exp in sorted(os.listdir(experiments_root)):
    result_path = os.path.join(experiments_root, exp, 'eval_results.json')
    if not os.path.isfile(result_path):
        continue
    with open(result_path, 'r') as f:
        r = json.load(f)
    rows.append({
        'Experiment' : r.get('experiment',      exp),
        'd_model'    : r['config'].get('d_model',   '?'),
        'n_heads'    : r['config'].get('n_heads',   '?'),
        'n_layers'   : r['config'].get('n_layers',  '?'),
        'ffn_dim'    : r['config'].get('ffn_dim',   '?'),
        'Perplexity' : r.get('test_perplexity', '?'),
        'BLEU%'      : r.get('bleu',            '?'),
        'ROUGE-1%'   : r.get('rouge1',          '?'),
        'ROUGE-2%'   : r.get('rouge2',          '?'),
        'ROUGE-L%'   : r.get('rougeL',          '?'),
        'Samples'    : r.get('test_samples',    '?'),
    })

if not rows:
    print('No eval_results.json found in any experiment folder yet.')
    print('Run Cells 3-5 for each experiment to populate this table.')
else:
    cols  = list(rows[0].keys())
    col_w = {c: max(len(c), max(len(str(r[c])) for r in rows)) + 2 for c in cols}
    header = ''.join(c.ljust(col_w[c]) for c in cols)
    sep    = '-' * len(header)
    print('\n' + sep)
    print('  EXPERIMENT COMPARISON SUMMARY')
    print(sep)
    print(header)
    print(sep)
    perps = [x['Perplexity'] for x in rows if isinstance(x['Perplexity'], (int, float))]
    for r in rows:
        marker = ' <- best' if r['Perplexity'] == min(perps) else ''
        print(''.join(str(r[c]).ljust(col_w[c]) for c in cols) + marker)
    print(sep)
    print(f'  Total experiments evaluated: {len(rows)}')
    print(sep)


--------------------------------------------------------------------------------------------------------------------------
  EXPERIMENT COMPARISON SUMMARY
--------------------------------------------------------------------------------------------------------------------------
Experiment                d_model  n_heads  n_layers  ffn_dim  Perplexity  BLEU%   ROUGE-1%  ROUGE-2%  ROUGE-L%  Samples  
--------------------------------------------------------------------------------------------------------------------------
exp1_heads4_layers6       256      4        6         1024     25.3389     2.7105  22.9796   4.0597    14.36     10130    
exp2_heads4_layers8       256      4        8         1024     23.6689     2.8066  23.0615   4.0799    14.4144   10130    
exp3_d512_heads4_layers8  512      4        8         2048     18.5726     3.1699  23.4861   4.3753    14.712    10130     <- best
exp4_d512_heads2_layers8  512      2        8         2048     19.1192     3.1682  23.5538   4.441

## Cell 7 — Manual Examples: Generated vs Reference
Picks 5 test samples and shows the patient question, reference doctor answer, and model-generated answer side by side.

In [36]:
print('=' * 70)
print(f'  MANUAL EVALUATION — {EXP_NAME}')
print('=' * 70)

sample_indices = [0, 100, 500, 1000, 2000]

for idx in sample_indices:
    tokens, masks = test_dataset[idx]
    token_list    = tokens.tolist()

    if DOCTOR_ID not in token_list:
        continue
    doctor_pos = token_list.index(DOCTOR_ID)

    patient_ids  = token_list[1:doctor_pos]  # skip <patient> token
    patient_text = sp.Decode([t for t in patient_ids if t not in [PAD_ID, EOS_ID]])

    reference_ids = []
    for t in token_list[doctor_pos + 1:]:
        if t in [PAD_ID, EOS_ID]:
            break
        reference_ids.append(t)
    reference_text = sp.Decode(reference_ids)

    patient_prompt = token_list[:doctor_pos + 1]
    generated_ids  = generate_for_eval(patient_prompt)
    generated_ids  = generated_ids[doctor_pos + 1:]
    generated_text = sp.Decode(generated_ids)

    print(f'\n--- Sample {idx} ---')
    print(f'Patient   : {patient_text[:200]}')
    print(f'Reference : {reference_text[:200]}')
    print(f'Generated : {generated_text[:200]}')
    print('-' * 70)

  MANUAL EVALUATION — exp4_d512_heads2_layers8

--- Sample 0 ---
Patient   : <patient> What causes dental root coming out from the gums after tooth extraction? all and one,I just had a tooth extraction done Monday just gone. Was glad to find that I don't have a dry socket but 
Reference : thanks for the consult ..Was extracted tooth -molar or premolar and was it very difficult to extract?? Since you tnk 2 roots are left bend,, firstly let the infection in the region come down..hope you
Generated : The pain,swelling and pus discharge could be because of the gum infection secondary to the deposits. Consult a oral physician and get a radiograph done to rule the amount of space present for the toot
----------------------------------------------------------------------

--- Sample 100 ---
Patient   : <patient> Reason for bruising and pain in the arm where the injection was administered? Received an injection in my left arm almost 4 weeks ago and I am now getting constant pain when I move it